In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Hyperparameters
input_size = 784      # 28x28
hidden_size = 128
num_classes = 10
num_epochs = 5
batch_size = 64
learning_rate = 0.001

# 3. MNIST dataset and loader
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root='data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='data', train=False, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# 4. Define a simple feedforward neural network
class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = x.view(-1, 28*28)     # Flatten the image
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = NeuralNet(input_size, hidden_size, num_classes).to(device)

# 5. Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 6. Training loop
for epoch in range(num_epochs):
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (batch_idx+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

# 7. Evaluation
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy of the model on the 10,000 test images: {100 * correct / total:.2f}%')


100%|██████████| 9.91M/9.91M [00:00<00:00, 16.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 498kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.53MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.16MB/s]


Epoch [1/5], Step [100/938], Loss: 0.4296
Epoch [1/5], Step [200/938], Loss: 0.2963
Epoch [1/5], Step [300/938], Loss: 0.2353
Epoch [1/5], Step [400/938], Loss: 0.2869
Epoch [1/5], Step [500/938], Loss: 0.2161
Epoch [1/5], Step [600/938], Loss: 0.3302
Epoch [1/5], Step [700/938], Loss: 0.1605
Epoch [1/5], Step [800/938], Loss: 0.2883
Epoch [1/5], Step [900/938], Loss: 0.1838
Epoch [2/5], Step [100/938], Loss: 0.2367
Epoch [2/5], Step [200/938], Loss: 0.2004
Epoch [2/5], Step [300/938], Loss: 0.1263
Epoch [2/5], Step [400/938], Loss: 0.1203
Epoch [2/5], Step [500/938], Loss: 0.0888
Epoch [2/5], Step [600/938], Loss: 0.1321
Epoch [2/5], Step [700/938], Loss: 0.0898
Epoch [2/5], Step [800/938], Loss: 0.1478
Epoch [2/5], Step [900/938], Loss: 0.1316
Epoch [3/5], Step [100/938], Loss: 0.1126
Epoch [3/5], Step [200/938], Loss: 0.0400
Epoch [3/5], Step [300/938], Loss: 0.1953
Epoch [3/5], Step [400/938], Loss: 0.1345
Epoch [3/5], Step [500/938], Loss: 0.0683
Epoch [3/5], Step [600/938], Loss: